# 5 - Which XLM-R layers make good emotion representations?This notebook runs the study in [`layerprobe/`](../layerprobe) and walks throughits output. The methodology, the metric choices and the limitations are in[`docs/LAYERWISE_EMOTION.md`](../docs/LAYERWISE_EMOTION.md); this notebook isthe operational side.**Order of business**1. Setup (Colab or local)2. A one-minute offline smoke run, so you can see every output before paying for a GPU3. The real run on a multilingual emotion corpus with XLM-R4. Reading the results: layer curves, combinations, scalar-mix weights5. Diagnostics: what makes a layer transfer6. Significance testing before you claim anything

## 1. SetupOn Colab, clone the repository and install the dependencies. Locally, just runthe cell after this one from the repository root.Pick a GPU runtime: feature extraction is the only expensive step, and it isabout twenty times faster on one.

In [ ]:
# Colab only -- skip if you are running this locally from the repo root.IN_COLAB = "google.colab" in str(get_ipython()) if "get_ipython" in dir() else Falseif IN_COLAB:    !git clone https://github.com/Tsegaye-misikir/NCC.git    %cd NCC    !pip install -q -r requirements.txt

In [ ]:
import os, sysfrom pathlib import Path# Make sure we are at the repository root, wherever the notebook was opened from.if not Path("layerprobe").exists() and Path("../layerprobe").exists():    os.chdir("..")sys.path.insert(0, str(Path.cwd()))import numpy as npimport pandas as pdimport matplotlib.pyplot as pltfrom layerprobe.config import load_configfrom layerprobe.pipeline import run_experimentpd.set_option("display.width", 160)pd.set_option("display.max_columns", 40)print("working directory:", Path.cwd())

## 2. Smoke run (offline, ~1 minute)`configs/smoke.yaml` uses a synthetic corpus and a synthetic encoder, so itdownloads nothing. Run it first: it produces exactly the same set of outputfiles as the real study, so you can check your plotting and reading codebefore spending GPU time.**The numbers it produces are not findings.** The synthetic encoder has alanguage-neutrality curve written into it by hand, to exercise the code paths.

In [ ]:
smoke_cfg = load_config("configs/smoke.yaml")smoke = run_experiment(smoke_cfg, verbose=True)print()print(open(smoke["summary_path"], encoding="utf-8").read())

## 3. The real study`configs/brighter.yaml` targets the SemEval-2025 Task 11 / BRIGHTERmulti-label emotion corpora with `xlm-roberta-base`, training zero-shot probeson English and evaluating on the low-resource targets.**Before the first run**, check `hf_path` and `hf_name_template` against thedataset card for the copy you have access to -- dataset ids and per-languageconfig names move, and a few languages annotate a different emotion set. Amismatch shows up as an all-zero column in `data_summary.csv`.Edit the config in place, or override from Python as below.

In [ ]:
cfg = load_config("configs/brighter.yaml")# Trim for a first pass; drop these lines for the full study.cfg.data.languages = ["eng", "amh", "hau"]cfg.data.source_languages = ["eng"]cfg.data.max_train_per_language = 1500cfg.seeds = [13, 42, 1234]print("encoder :", cfg.encoder.model_name)print("languages:", cfg.data.languages, "-> targets", cfg.data.resolved_target_languages())print("seeds   :", cfg.seeds)

In [ ]:
# Feature extraction dominates the runtime and is cached under cfg.cache_dir,# so re-running this cell after changing only probe settings is fast.payload = run_experiment(cfg, verbose=True)results = pd.DataFrame(payload["results"])results.head()

### Sanity check the data firstBefore reading a single F1: are the label columns populated, and are thesplits the size you expect? An emotion column that is all zeros means thedataset's column names do not match `data.emotions`.

In [ ]:
pd.DataFrame(payload["data_summary"])

## 4. Reading the results### 4a. The layer curveThe main figure. If the curve peaks well before the final layer, the defaultchoice was costing you accuracy -- and the gap is usually widest for thezero-shot targets.

In [ ]:
def layer_curve(results, experiment):    frame = results[(results.experiment == experiment) & (results.kind == "single")].copy()    frame["layer"] = frame["layers"].apply(lambda v: v[0])    return frame.drop_duplicates(["eval_language", "layer"]).sort_values("layer")fig, axes = plt.subplots(1, 2, figsize=(13, 4.5), sharey=True)for ax, experiment in zip(axes, ["monolingual", "zeroshot"]):    frame = layer_curve(results, experiment)    for language, group in frame.groupby("eval_language"):        ax.errorbar(group.layer, group.macro_f1_mean, yerr=group.macro_f1_std,                    marker="o", capsize=3, label=language)    if len(frame):        ax.axvline(frame.layer.max(), color="grey", ls="--", lw=1)    ax.set_title(experiment); ax.set_xlabel("encoder layer"); ax.grid(alpha=.3); ax.legend()axes[0].set_ylabel("macro-F1")plt.tight_layout()

### 4b. Best combination per setting`gain_over_last` is the headline number: how much macro-F1 the defaultfinal-layer representation was leaving on the table.

In [ ]:
best = pd.DataFrame(payload["best"])cols = ["experiment", "eval_language", "best_combination", "best_macro_f1",        "last_layer_macro_f1", "gain_over_last"]best[[c for c in cols if c in best.columns]].sort_values(["experiment", "eval_language"])

### 4c. What to actually recommendA combination that wins in one language and collapses in another is not arecommendation. Mean rank across evaluation languages is what survives.

In [ ]:
from layerprobe.reporting import layer_rankingranking = layer_ranking(payload["results"])for experiment, group in ranking.groupby("experiment"):    print(f"--- {experiment} ---")    print(group.head(8)[["combination", "kind", "mean_rank", "mean_macro_f1"]].to_string(index=False))    print()

### 4d. What the scalar mix asks forThe learned softmax weights over layers, found jointly with the classifierrather than by grid search. They should broadly agree with the single-layercurve; if they do not, suspect that one layer's activation norm is dominatingand check `scalar_mix.layer_norm`.

In [ ]:
mix_rows = [r for r in payload["results"] if r["combination"] == "scalar_mix" and r.get("layer_weights_mean")]fig, ax = plt.subplots(figsize=(8, 4.5))for row in mix_rows:    ax.plot(row["layers"], row["layer_weights_mean"], marker="o", alpha=.85,            label=f"{row['experiment']}/{row['eval_language']}")ax.set_xlabel("encoder layer"); ax.set_ylabel("learned weight")ax.set_title("Scalar-mix weights"); ax.grid(alpha=.3); ax.legend(fontsize=7, ncol=2)plt.tight_layout()

### 4e. Negative transfer`transfer_gap` is in-language macro-F1 minus zero-shot macro-F1, so positivemeans training on English cost you. The question is whether choosing a betterlayer narrows it.

In [ ]:
zs = results[(results.experiment == "zeroshot") & results.transfer_gap.notna()]gap = (zs.pivot_table(index="combination", columns="eval_language", values="transfer_gap")         .assign(mean_gap=lambda d: d.mean(axis=1))         .sort_values("mean_gap"))print("Smallest transfer gaps (best) at the top; 'last' is the default:")gap.head(10)

In [ ]:
# Did any combination fail to beat the majority-class baseline? If so it# transferred nothing at all, whatever its F1 looks like.failed = zs[zs.above_majority <= 0][["eval_language", "combination", "macro_f1_mean", "majority_macro_f1"]]print(f"{len(failed)} of {len(zs)} zero-shot configurations did not clear the majority baseline")failed.head()

## 5. Diagnostics: why a layer transfersTwo explanatory measurements, both per layer:- **language identifiability** -- can a linear probe tell which language a  sentence is in, from the same vector? High means the layer keeps  language-specific directions that a cross-lingual probe can wrongly latch onto.- **cross-lingual CKA** -- how similarly does the layer organise the two  languages' sentences?

In [ ]:
lang_probe = pd.DataFrame(payload.get("language_probe", []))alignment = pd.DataFrame(payload.get("alignment", []))fig, axes = plt.subplots(1, 2, figsize=(13, 4))if len(lang_probe):    axes[0].plot(lang_probe.layer, lang_probe.language_id_accuracy, marker="o", color="#c44e52")    axes[0].plot(lang_probe.layer, lang_probe.chance, ls="--", color="grey", label="chance")    axes[0].legend()axes[0].set_title("Language identifiability"); axes[0].set_xlabel("layer"); axes[0].grid(alpha=.3)if len(alignment):    for (src, tgt), group in alignment.groupby(["source", "target"]):        axes[1].plot(group.layer, group.cka, marker="o", label=f"{src}->{tgt}")    axes[1].legend(fontsize=8)axes[1].set_title("Cross-lingual CKA"); axes[1].set_xlabel("layer"); axes[1].grid(alpha=.3)plt.tight_layout()

In [ ]:
# Do the diagnostics actually predict transfer? Note n_layers: these# correlations rest on ~13 points and are a hypothesis, not a result.payload.get("diagnostics_correlation", {})

## 6. Before you claim anything: significanceWith three seeds, a one-point macro-F1 gap is usually noise. `paired_bootstrap`takes the per-seed scores of two configurations and returns a two-sidedp-value for the difference.

In [ ]:
from layerprobe.metrics import paired_bootstrapdef compare(results, experiment, language, combo_a, combo_b="last"):    rows = results[(results.experiment == experiment) & (results.eval_language == language)]    a = rows[rows.combination == combo_a].iloc[0]["macro_f1_per_seed"]    b = rows[rows.combination == combo_b].iloc[0]["macro_f1_per_seed"]    return {        "language": language, combo_a: float(np.mean(a)), combo_b: float(np.mean(b)),        "diff": float(np.mean(a) - np.mean(b)), "p": paired_bootstrap(a, b),    }targets = cfg.data.resolved_target_languages()comparisons = []for language in targets:    winner = best[(best.experiment == "zeroshot") & (best.eval_language == language)]    if len(winner):        comparisons.append(compare(results, "zeroshot", language, winner.iloc[0]["best_combination"]))pd.DataFrame(comparisons)

A caution on that table: the winning combination was *chosen* by looking atthese same test scores, so its p-value is optimistic. To make a clean claim,pick the combination on dev (`dev_macro_f1_mean` in `results.csv`) or on aheld-out set of languages, and only then test it here.## 7. Robustness checks worth runningEach of these is one cell and re-uses the cached features where it can:- **Pooling** -- `cfg.encoder.pooling = "cls"`. Some of any layer difference  is a difference in how well that layer's geometry survives averaging.- **Probe capacity** -- `cfg.probe.kind = "mlp"`. Does the layer ranking hold  for a non-linear probe?- **Encoder** -- `cfg.encoder.model_name = "xlm-roberta-large"` (24 layers, so  the depth profile shifts) or `bert-base-multilingual-cased`.- **Training size** -- sweep `cfg.data.max_train_per_language`. The best layer  often moves with how much data the probe has.Every run writes to its own `output_dir`, so results do not overwrite.

In [ ]:
# Example: does the ranking survive a switch to CLS pooling?cls_cfg = load_config("configs/brighter.yaml")cls_cfg.data.languages = cfg.data.languagescls_cfg.data.source_languages = cfg.data.source_languagescls_cfg.data.max_train_per_language = cfg.data.max_train_per_languagecls_cfg.seeds = cfg.seedscls_cfg.encoder.pooling = "cls"cls_cfg.output_dir = "results/brighter-xlmr-cls"# cls_payload = run_experiment(cls_cfg, verbose=True)   # uncomment to run